# Gradient-guided model experiments

In [5]:
import os, sys, json
import numpy as np
import torch
import matplotlib.pyplot as plt
from functools import partial
from diffusers.models.unets.unet_2d import UNet2DModel
from diffusers.training_utils import EMAModel

from utils import (
    NpyImageDataset, channel_normalize, channel_denormalize,
    generate_satellite_track_mask, get_device,
    make_plot, make_difference_plot,
)
from gradient_based.sampler import Sampler

In [ ]:
CHECKPOINT_DIR = "/home/checkpoints"
RUN_NAME       = "best_model/"                         
DATA_ROOT      = "/mnt/sciml/a.sadreev/sea_ice_data"

N_SAMPLES      = 25
NUM_TIMESTEPS  = 50
GUIDANCE_SCALE = 1.0
N_TRACKS_RANGE = (1, 4)
IMAGE_SIZE     = (320, 256)

DEVICE = get_device()
print(f"device: {DEVICE}")

with open(os.path.join(DATA_ROOT, "train", "stats.json")) as f:
    stats = json.load(f)
CHANNEL_MEAN = tuple(stats["mean"])
CHANNEL_STD  = tuple(stats["std"])
print(f"channel_mean={CHANNEL_MEAN}  channel_std={CHANNEL_STD}")

device: cuda
channel_mean=(0.1158, 0.1153)  channel_std=(0.3008, 0.3501)


## Загрузка модели

In [7]:
if RUN_NAME is None:
    runs = sorted(d for d in os.listdir(CHECKPOINT_DIR) if d.startswith("run_"))
    assert runs, f"Нет run_* в {CHECKPOINT_DIR}"
    RUN_NAME = runs[-1]
run_dir = os.path.join(CHECKPOINT_DIR, RUN_NAME)
print(f"Используем run: {run_dir}")

cfg_path = os.path.join(run_dir, "config.json")
if os.path.exists(cfg_path):
    with open(cfg_path) as f:
        saved_cfg = json.load(f)
    print(json.dumps({k: v for k, v in saved_cfg.items()
                      if k not in ("channel_mean", "channel_std")}, indent=2))

Используем run: /home/checkpoints/best_model/


In [ ]:
model = UNet2DModel(
    sample_size=IMAGE_SIZE,
    in_channels=4,
    out_channels=2,
    layers_per_block=2,
    block_out_channels=(64, 128, 256, 512, 512),
    down_block_types=("DownBlock2D", "DownBlock2D", "DownBlock2D",
                      "AttnDownBlock2D", "DownBlock2D"),
    up_block_types=("UpBlock2D", "AttnUpBlock2D", "UpBlock2D",
                    "UpBlock2D", "UpBlock2D"),
)

ema_path = os.path.join(run_dir, "ema_state.pth")
assert os.path.exists(ema_path), f"Файл не найден: {ema_path}"

ema = EMAModel(model.parameters(), decay=0.999)
ema.load_state_dict(torch.load(ema_path, map_location="cpu", weights_only=True))
ema.copy_to(model.parameters())

model.eval().to(DEVICE)
sampler = Sampler(model)
print("Модель загружена.")

AssertionError: Файл не найден: /home/checkpoints/best_model/ema_best_model.pth

## Валидационные данные

In [ ]:
transform = partial(channel_normalize, channel_mean=CHANNEL_MEAN, channel_std=CHANNEL_STD)

val_dataset = NpyImageDataset(
    folder=os.path.join(DATA_ROOT, "valid"),
    transform=transform,
    preload=False,
    mmap_mode='r',
)
print(f"Всего val-сэмплов: {len(val_dataset)}")

indices = np.linspace(0, len(val_dataset) - 1, N_SAMPLES, dtype=int)
print(f"Выбраны индексы: {indices}")

clean_images = torch.stack([val_dataset[i] for i in indices]).to(DEVICE)
print(f"clean_images shape: {clean_images.shape}")

## Загрузка маски спутникового трека

In [ ]:
satellite_data_path = "/mnt/sciml/data_assimilation/sral_si/sral_new_format/"

all_files = sorted([f for f in os.listdir(satellite_data_path)
                    if os.path.isfile(os.path.join(satellite_data_path, f))])

first_file = all_files[1500]
full_path = os.path.join(satellite_data_path, first_file)

satellite_data_example = np.load(full_path)
satellite_data_example = np.pad(
    satellite_data_example[0], ((4, 5), (15, 16)),
    mode='constant', constant_values=((None, None), (None, None))
)
satellite_data_mask = torch.from_numpy(
    np.where(np.isnan(satellite_data_example), 0, 1)
).to(DEVICE)
print(satellite_data_mask.shape)

plt.imshow(satellite_data_mask.cpu())
plt.colorbar()
plt.show()

## Inference — gradient-guided сэмплинг

In [ ]:
mask_batched = satellite_data_mask.unsqueeze(0).unsqueeze(1)          # (1, 1, H, W)
mask_batched = mask_batched.expand(N_SAMPLES, -1, -1, -1)             # (N, 1, H, W)

observed = clean_images * satellite_data_mask                          # (N, 2, H, W)

predictions = sampler.sample_with_condition(
    y=observed,
    mask=mask_batched,
    num_timesteps=NUM_TIMESTEPS,
    guidance_scale=GUIDANCE_SCALE,
    device=DEVICE,
    batch_size=N_SAMPLES,
)

predictions = predictions.cpu()
print(f"predictions shape: {predictions.shape}")

## Визуализация

In [ ]:
clean_cpu    = clean_images.cpu()
pred_cpu     = predictions.cpu()
observed_cpu = clean_cpu * satellite_data_mask.cpu()

make_plot(clean_cpu,    CHANNEL_MEAN, CHANNEL_STD, N_SAMPLES, title="Truth — Concentration (channel 0)")
make_plot(observed_cpu, CHANNEL_MEAN, CHANNEL_STD, N_SAMPLES, title="Observed on tracks — Concentration (channel 0)")
make_plot(pred_cpu,     CHANNEL_MEAN, CHANNEL_STD, N_SAMPLES, title="Prediction — Concentration (channel 0)")

In [ ]:
data_example = np.load(r"/mnt/sciml/data_assimilation/da_arctic_2015-2024_v0.1/preds/ocean+atmosphere_24_2015-01-17.npy")
water_mask_initial = np.where(np.isnan(data_example[7, 0, :, :]), 0, 1)
water_mask = np.pad(water_mask_initial, ((4, 5), (15, 16)))
plt.imshow(water_mask)
plt.colorbar()
plt.show()

In [ ]:
make_difference_plot(clean_cpu, pred_cpu, CHANNEL_MEAN, CHANNEL_STD, N_SAMPLES, land_mask=water_mask)

## Средний MSE по треку

In [ ]:
truth_dn = channel_denormalize(clean_images.clone().cpu(), CHANNEL_MEAN, CHANNEL_STD)
pred_dn  = channel_denormalize(predictions.clone().cpu(), CHANNEL_MEAN, CHANNEL_STD)
mask_cpu = satellite_data_mask.cpu()  # (H, W)

diff2    = (truth_dn - pred_dn) ** 2
mask_exp = mask_cpu.unsqueeze(0).unsqueeze(0)

n_pixels = mask_cpu.sum().item()
mse_per_sample = (diff2 * mask_exp).sum(dim=(2, 3)) / n_pixels

print(f"{'Sample':>8}  {'Concentration MSE':>18}  {'Thickness MSE':>14}")
print("-" * 46)
for i in range(N_SAMPLES):
    print(f"  #{indices[i]:>5}  {mse_per_sample[i,0].item():>18.5f}  {mse_per_sample[i,1].item():>14.5f}")
print("-" * 46)
print(f"  {'mean':>6}  {mse_per_sample[:,0].mean().item():>18.5f}  {mse_per_sample[:,1].mean().item():>14.5f}")